# Polymarket Dashboard

Explore active events and markets on [Polymarket](https://polymarket.com) using the [Gamma API](https://docs.polymarket.com/market-data/fetching-markets).

| Section | Description |
|---------|-------------|
| 1 | Top events by 24h volume |
| 2 | Top events by total volume |
| 3 | Top markets by 24h volume |
| 4 | Top markets by total volume |
| 5 | Biggest price movers (24h) |

In [4]:
import requests
import pandas as pd
import json
from concurrent.futures import ThreadPoolExecutor, as_completed

GAMMA_API = "https://gamma-api.polymarket.com"
CLOB_API = "https://clob.polymarket.com"

def fetch_events(order="volume24hr", limit=100):
    resp = requests.get(f"{GAMMA_API}/events", params={
        "active": "true",
        "closed": "false",
        "order": order,
        "ascending": "false",
        "limit": limit,
        "include_tag": "true",
    })
    resp.raise_for_status()
    return resp.json()

def fetch_markets(order="volume24hr", limit=100):
    resp = requests.get(f"{GAMMA_API}/markets", params={
        "active": "true",
        "closed": "false",
        "order": order,
        "ascending": "false",
        "limit": limit,
        "include_tag": "true",
    })
    resp.raise_for_status()
    return resp.json()

def events_df(events, n=10):
    rows = []
    for e in events[:n]:
        rows.append({
            "Title": e.get("title", "N/A"),
            "24h Volume": f"${float(e.get('volume24hr', 0)):,.0f}",
            "Total Volume": f"${float(e.get('volume', 0)):,.0f}",
            "Liquidity": f"${float(e.get('liquidity', 0)):,.0f}",
            "# Markets": len(e.get("markets", [])),
            "Slug": e.get("slug", ""),
        })
    df = pd.DataFrame(rows)
    df.index = range(1, len(df) + 1)
    df.index.name = "Rank"
    return df

def markets_df(markets, n=10):
    rows = []
    for m in markets[:n]:
        prices = m.get("outcomePrices", "[]") or "[]"
        rows.append({
            "Question": m.get("question", "N/A"),
            "24h Volume": f"${float(m.get('volume24hr', 0) or 0):,.0f}",
            "Total Volume": f"${float(m.get('volume', 0) or 0):,.0f}",
            "Liquidity": f"${float(m.get('liquidity', 0) or 0):,.0f}",
            "Outcome Prices": prices,
            "Slug": m.get("slug", ""),
        })
    df = pd.DataFrame(rows)
    df.index = range(1, len(df) + 1)
    df.index.name = "Rank"
    return df

def _get_tag_labels(item):
    return {t.get("label", "") for t in (item.get("tags") or [])}

def exclude_tags(items, tags):
    """Remove items that have any of the given tags."""
    tags = set(tags)
    return [i for i in items if not (_get_tag_labels(i) & tags)]

def keep_tags(items, tags):
    """Keep only items that have at least one of the given tags."""
    tags = set(tags)
    return [i for i in items if _get_tag_labels(i) & tags]

def get_yes_token_id(market):
    """Extract the Yes token ID (first entry) from clobTokenIds."""
    ids_str = market.get("clobTokenIds", "[]") or "[]"
    ids = json.loads(ids_str)
    return ids[0] if ids else None

def fetch_price_ago(token_id, interval="1d", timeout=10):
    """Fetch the earliest price point in the given interval from the CLOB API.
    interval: '1h', '6h', '1d', '1w', '1m'
    """
    fidelity = {"1h": 1, "6h": 5, "1d": 60, "1w": 360, "1m": 1440}.get(interval, 60)
    try:
        resp = requests.get(f"{CLOB_API}/prices-history", params={
            "market": token_id,
            "interval": interval,
            "fidelity": fidelity,
        }, timeout=timeout)
        resp.raise_for_status()
        history = resp.json().get("history", [])
        if history:
            return history[0]["p"]
    except Exception:
        pass
    return None

print("Helpers loaded.")

Helpers loaded.


---
## Section 1: Top Events by 24h Volume

In [5]:
events_by_24h = fetch_events(order="volume24hr")
print(f"Fetched {len(events_by_24h)} events (ordered by 24h volume)")
events_df(events_by_24h)

Fetched 100 events (ordered by 24h volume)


,Title,24h Volume,Total Volume,Liquidity,# Markets,Slug
Rank,,,,,,
1,Khamenei out as Supreme Leader of Iran by Febr...,"$62,854,218","$84,943,949","$6,746,123",1,khamenei-out-as-supreme-leader-of-iran-by-febr...
2,Khamenei out as Supreme Leader of Iran by Marc...,"$30,570,127","$51,673,827","$3,236,644",1,khamenei-out-as-supreme-leader-of-iran-by-marc...
3,2026 FIFA World Cup Winner,"$7,806,679","$228,627,104","$38,969,055",60,2026-fifa-world-cup-winner-595
4,Who will Trump nominate as Fed Chair?,"$7,151,506","$552,861,576","$50,347,615",39,who-will-trump-nominate-as-fed-chair
5,Fed decision in March?,"$5,747,363","$192,448,346","$4,665,290",4,fed-decision-in-march-885
6,Democratic Presidential Nominee 2028,"$5,100,013","$736,530,716","$38,312,763",128,democratic-presidential-nominee-2028
7,Presidential Election Winner 2028,"$4,326,996","$346,687,107","$17,663,597",128,presidential-election-winner-2028
8,Republican Presidential Nominee 2028,"$4,171,615","$340,773,790","$17,835,430",128,republican-presidential-nominee-2028
9,Will the Iranian regime fall by March 31?,"$3,766,942","$11,653,727","$352,393",1,will-the-iranian-regime-fall-by-march-31


---
## Section 2: Top Events by Total Volume

In [6]:
events_by_total = fetch_events(order="volume")
print(f"Fetched {len(events_by_total)} events (ordered by total volume)")
events_df(events_by_total)

Fetched 100 events (ordered by total volume)


,Title,24h Volume,Total Volume,Liquidity,# Markets,Slug
Rank,,,,,,
1,Democratic Presidential Nominee 2028,"$5,100,013","$736,530,716","$38,312,763",128,democratic-presidential-nominee-2028
2,Who will Trump nominate as Fed Chair?,"$7,151,506","$552,861,576","$50,347,615",39,who-will-trump-nominate-as-fed-chair
3,Presidential Election Winner 2028,"$4,326,996","$346,687,107","$17,663,597",128,presidential-election-winner-2028
4,Republican Presidential Nominee 2028,"$4,171,615","$340,773,790","$17,835,430",128,republican-presidential-nominee-2028
5,2026 NBA Champion,"$2,516,133","$307,264,114","$16,571,061",30,2026-nba-champion
6,English Premier League Winner,"$479,380","$276,373,783","$19,707,744",25,english-premier-league-winner
7,UEFA Champions League Winner,"$1,185,491","$253,418,010","$6,280,831",60,uefa-champions-league-winner
8,2026 FIFA World Cup Winner,"$7,806,679","$228,627,104","$38,969,055",60,2026-fifa-world-cup-winner-595
9,Fed decision in March?,"$5,747,363","$192,448,346","$4,665,290",4,fed-decision-in-march-885


---
## Section 3: Top Markets by 24h Volume

In [7]:
markets_by_24h = fetch_markets(order="volume24hr")
print(f"Fetched {len(markets_by_24h)} markets (ordered by 24h volume)")
markets_df(markets_by_24h)

Fetched 100 markets (ordered by 24h volume)


,Question,24h Volume,Total Volume,Liquidity,Outcome Prices,Slug
Rank,,,,,,
1,Khamenei out as Supreme Leader of Iran by Febr...,"$62,854,218","$84,953,370","$6,755,938","[""0.9985"", ""0.0015""]",khamenei-out-as-supreme-leader-of-iran-by-febr...
2,Khamenei out as Supreme Leader of Iran by Marc...,"$30,570,127","$51,684,246","$3,228,074","[""0.9985"", ""0.0015""]",khamenei-out-as-supreme-leader-of-iran-by-marc...
3,Will the Iranian regime fall by March 31?,"$3,766,942","$11,659,108","$349,434","[""0.2225"", ""0.7775""]",will-the-iranian-regime-fall-by-march-31
4,Will Trump nominate Judy Shelton as the next F...,"$3,416,311","$102,010,331","$879,460","[""0.0385"", ""0.9615""]",will-trump-nominate-judy-shelton-as-the-next-f...
5,Will the Fed decrease interest rates by 50+ bp...,"$2,248,890","$78,052,484","$1,736,030","[""0.0055"", ""0.9945""]",will-the-fed-decrease-interest-rates-by-50-bps...
6,Will South Africa win the 2026 FIFA World Cup?,"$1,762,318","$13,430,915","$1,129,246","[""0.0015"", ""0.9985""]",will-south-africa-win-the-2026-fifa-world-cup
7,Will there be no change in Fed interest rates ...,"$1,458,935","$23,820,688","$582,578","[""0.9595"", ""0.0405""]",will-there-be-no-change-in-fed-interest-rates-...
8,Will Iran close the Strait of Hormuz by March 31?,"$1,338,355","$2,175,433","$40,935","[""0.44"", ""0.56""]",will-iran-close-the-strait-of-hormuz-by-march-31
9,Will the Fed increase interest rates by 25+ bp...,"$1,217,852","$66,837,545","$1,844,205","[""0.0035"", ""0.9965""]",will-the-fed-increase-interest-rates-by-25-bps...


In [8]:
print(json.dumps(events_by_24h[4], indent=2))

{
  "id": "67284",
  "ticker": "fed-decision-in-march-885",
  "slug": "fed-decision-in-march-885",
  "title": "Fed decision in March?",
  "description": "The FED interest rates are defined in this market by the upper bound of the target federal funds range. The decisions on the target federal fund range are made by the Federal Open Market Committee (FOMC) meetings.\n\nThis market will resolve to the amount of basis points the upper bound of the target federal funds rate is changed by versus the level it was prior to the Federal Reserve's March 2026 meeting.\n\nIf the target federal funds rate is changed to a level not expressed in the displayed options, the change will be rounded up to the nearest 25 and will resolve to the relevant bracket. (e.g. if there's a cut/increase of 12.5 bps it will be considered to be 25 bps)\n\nThe resolution source for this market is the FOMC\u2019s statement after its meeting scheduled for March 17 - 18, 2026 according to the official calendar: https://ww

In [9]:
print(json.dumps(markets_by_24h[2], indent=2))

{
  "id": "958442",
  "question": "Will the Iranian regime fall by March 31?",
  "conditionId": "0x61ce3773237a948584e422de72265f937034af418a8b703e3a860ea62e59ff36",
  "slug": "will-the-iranian-regime-fall-by-march-31",
  "resolutionSource": "",
  "endDate": "2026-03-31T00:00:00Z",
  "liquidity": "349434.37305",
  "startDate": "2025-12-17T23:01:53.379097Z",
  "image": "https://polymarket-upload.s3.us-east-2.amazonaws.com/will-the-iranian-regime-fall-in-2025-YLXIniTmQs4q.png",
  "icon": "https://polymarket-upload.s3.us-east-2.amazonaws.com/will-the-iranian-regime-fall-in-2025-YLXIniTmQs4q.png",
  "description": "This market will resolve to \"Yes\" if the Islamic Republic of Iran\u2019s current ruling regime is overthrown, collapsed, or otherwise ceases to govern by March 31, 2026, 11:59 PM ET. Otherwise, this market will resolve to \u201cNo\u201d.\n\nThis requires a broad consensus of reporting indicating that core structures of the Islamic Republic (e.g. the office of the Supreme Leade

---
## Section 4: Top Markets by Total Volume

In [10]:
markets_by_total = fetch_markets(order="volumeNum")
print(f"Fetched {len(markets_by_total)} markets (ordered by total volume)")
markets_df(markets_by_total)

Fetched 100 markets (ordered by total volume)


,Question,24h Volume,Total Volume,Liquidity,Outcome Prices,Slug
Rank,,,,,,
1,Will Trump nominate Judy Shelton as the next F...,"$3,416,311","$102,010,331","$879,460","[""0.0385"", ""0.9615""]",will-trump-nominate-judy-shelton-as-the-next-f...
2,Khamenei out as Supreme Leader of Iran by Febr...,"$62,854,218","$84,953,370","$6,755,938","[""0.9985"", ""0.0015""]",khamenei-out-as-supreme-leader-of-iran-by-febr...
3,Will the Fed decrease interest rates by 50+ bp...,"$2,248,890","$78,052,484","$1,736,030","[""0.0055"", ""0.9945""]",will-the-fed-decrease-interest-rates-by-50-bps...
4,Will the Fed increase interest rates by 25+ bp...,"$1,217,852","$66,837,545","$1,844,205","[""0.0035"", ""0.9965""]",will-the-fed-increase-interest-rates-by-25-bps...
5,Khamenei out as Supreme Leader of Iran by Marc...,"$30,570,127","$51,684,246","$3,228,074","[""0.9985"", ""0.0015""]",khamenei-out-as-supreme-leader-of-iran-by-marc...
6,Will the Indiana Pacers win the 2026 NBA Finals?,"$8,486","$47,833,397","$349,592","[""0.0005"", ""0.9995""]",will-the-indiana-pacers-win-the-2026-nba-finals
7,Will Trump nominate Kevin Warsh as the next Fe...,"$816,718","$44,662,230","$584,802","[""0.9335"", ""0.0665""]",will-trump-nominate-kevin-warsh-as-the-next-fe...
8,Will Chelsea Clinton win the 2028 Democratic p...,"$65,873","$40,357,088","$354,179","[""0.0095"", ""0.9905""]",will-chelsea-clinton-win-the-2028-democratic-p...
9,Will Trump nominate Scott Bessent as the next ...,"$312,859","$38,552,269","$1,378,163","[""0.0005"", ""0.9995""]",will-trump-nominate-scott-bessent-as-the-next-...


---
## Section 5: Biggest Price Movers

Use the dropdown to pick a lookback window (1 hour → 1 month). Fetches actual price history from the CLOB `/prices-history` endpoint and computes `current_price − price_then` in percentage points.

In [11]:
import ipywidgets as widgets
from IPython.display import display, clear_output

PERIODS = {"1 Hour": "1h", "6 Hours": "6h", "1 Day": "1d", "1 Week": "1w", "1 Month": "1m"}

all_markets = fetch_markets(order="volume24hr", limit=1000)
filtered = exclude_tags(all_markets, ["Crypto Prices", "Sports"])
candidates = [m for m in filtered if float(m.get("volume24hr", 0) or 0) > 0]
print(f"Loaded {len(candidates)} candidate markets (from {len(all_markets)} total)")

period_picker = widgets.Dropdown(
    options=list(PERIODS.keys()),
    value="1 Day",
    description="Lookback:",
    style={"description_width": "auto"},
)
out = widgets.Output()

def refresh(_=None):
    with out:
        clear_output(wait=True)
        interval = PERIODS[period_picker.value]
        print(f"Fetching {period_picker.value} price history for {len(candidates)} markets...")

        def _compute(m):
            token_id = get_yes_token_id(m)
            if not token_id:
                return None
            price_then = fetch_price_ago(token_id, interval=interval)
            if price_then is None:
                return None
            prices_str = m.get("outcomePrices", "[]") or "[]"
            prices = json.loads(prices_str)
            if not prices:
                return None
            current = float(prices[0])
            return {**m, "_change": current - price_then,
                    "_abs_change": abs(current - price_then),
                    "_current_yes": current}

        results = []
        with ThreadPoolExecutor(max_workers=20) as pool:
            for r in pool.map(_compute, candidates):
                if r is not None:
                    results.append(r)

        movers = sorted(results, key=lambda m: m["_abs_change"], reverse=True)

        rows = []
        for m in movers[:20]:
            rows.append({
                "Question": m.get("question", "N/A"),
                "Change": f"{m['_change']:+.0%}",
                "Current": f"{m['_current_yes']:.0%}",
                "24h Volume": f"${float(m.get('volume24hr', 0) or 0):,.0f}",
                "Outcome Prices": m.get("outcomePrices", "[]") or "[]",
                "Slug": m.get("slug", ""),
            })

        df = pd.DataFrame(rows)
        df.index = range(1, len(df) + 1)
        df.index.name = "Rank"
        print(f"Got history for {len(results)}/{len(candidates)} markets\n")
        print(f"Top 20 movers — {period_picker.value} window — out of {len(all_markets)} active markets")
        display(df)

period_picker.observe(refresh, names="value")
display(period_picker, out)
refresh()

Loaded 289 candidate markets (from 500 total)


Dropdown(description='Lookback:', index=2, options=('1 Hour', '6 Hours', '1 Day', '1 Week', '1 Month'), style=…

Output()